# Traffic Risk Assessment EDA

This notebook analyzes the US Accidents data using the same contract as the production pipeline:

- **before 2020** records are used for offline pretraining.
- **from 2020 onward** records are used only for realtime replay and online retraining simulation.
- Analysis cells compare both periods side by side, but modeling decisions are selected from the before-2020 split to avoid temporal leakage.

In [ ]:
from pathlib import Path
import sys
import csv
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make this notebook runnable from Jupyter, VS Code, or the repository root.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "processing").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from processing.feature_engineering import build_features

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Core paths. The processed before-2020 file is produced by ml/dataset/dataset_offline.py.
BEFORE_FEATURE_PATH = PROJECT_ROOT / "data/process/us_train_offline_before_2020.csv"
AFTER_RAW_PATH = PROJECT_ROOT / "data/split/us_pipeline_from_2020.csv"
RAW_FULL_PATH = PROJECT_ROOT / "data/raw/US.csv"

# Plotting millions of rows adds noise and makes notebooks hard to review.
# Tables use chunked full-data scans where practical; plots use stable samples.
MAX_PLOT_ROWS_PER_SPLIT = 300_000
RANDOM_SEED = 42

print("before_2020 feature file:", BEFORE_FEATURE_PATH, BEFORE_FEATURE_PATH.exists())
print("from_2020 replay file:", AFTER_RAW_PATH, AFTER_RAW_PATH.exists())
print("raw full file:", RAW_FULL_PATH, RAW_FULL_PATH.exists())

In [ ]:
def read_csv_sample(path: Path, nrows: int | None = None, usecols: list[str] | None = None) -> pd.DataFrame:
    """Read a bounded CSV sample for visualization while keeping notebook runtime stable."""
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")
    return pd.read_csv(path, nrows=nrows, usecols=usecols, low_memory=False)


def build_feature_sample_from_raw(path: Path, max_rows: int) -> pd.DataFrame:
    """
    Build an after-2020 feature sample through the production build_features() function.

    The point is not to invent EDA-only transformations. This keeps every chart aligned with
    Flink inference, Spark Gold generation, and H2O training schema.
    """
    rows = []
    with path.open("r", encoding="utf-8", newline="") as input_file:
        reader = csv.DictReader(input_file)
        for raw_row in reader:
            feature_row = build_features(raw_row)
            if feature_row is not None:
                rows.append(feature_row)
            if len(rows) >= max_rows:
                break
    return pd.DataFrame(rows)


before_df = read_csv_sample(BEFORE_FEATURE_PATH, MAX_PLOT_ROWS_PER_SPLIT)
after_df = build_feature_sample_from_raw(AFTER_RAW_PATH, MAX_PLOT_ROWS_PER_SPLIT)

before_df["period"] = "before_2020_pretrain"
after_df["period"] = "from_2020_realtime"
eda_df = pd.concat([before_df, after_df], ignore_index=True)

print("before_df shape:", before_df.shape)
print("after_df shape:", after_df.shape)
print("combined EDA sample shape:", eda_df.shape)
display(eda_df.head())

## Dataset Shape And Schema

This section checks that both periods expose the same engineered schema. Any mismatch here is more important than a model metric because a schema mismatch breaks realtime inference and online retraining.

In [ ]:
schema_table = pd.DataFrame({
    "column": sorted(set(before_df.columns) | set(after_df.columns)),
})
schema_table["in_before_2020"] = schema_table["column"].isin(before_df.columns)
schema_table["in_from_2020"] = schema_table["column"].isin(after_df.columns)
schema_table["before_dtype"] = schema_table["column"].map(lambda col: str(before_df[col].dtype) if col in before_df.columns else "missing")
schema_table["after_dtype"] = schema_table["column"].map(lambda col: str(after_df[col].dtype) if col in after_df.columns else "missing")

overview = pd.DataFrame([
    {"period": "before_2020_pretrain", "rows_in_sample": len(before_df), "columns": before_df.shape[1], "min_year": before_df["event_year"].min(), "max_year": before_df["event_year"].max()},
    {"period": "from_2020_realtime", "rows_in_sample": len(after_df), "columns": after_df.shape[1], "min_year": after_df["event_year"].min(), "max_year": after_df["event_year"].max()},
])

display(overview)
display(schema_table)

## Temporal Distribution

The offline split must contain only records before 2020. The realtime replay split must contain records from 2020 onward. The charts below make the temporal split explicit for reviewers.

In [ ]:
year_counts = (
    eda_df.groupby(["period", "event_year"])
    .size()
    .reset_index(name="accident_count")
    .sort_values(["period", "event_year"])
)

display(year_counts)

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=False)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    data = year_counts[year_counts["period"] == period]
    sns.barplot(data=data, x="event_year", y="accident_count", ax=axis, color="#4C78A8")
    axis.set_title(f"Accident Count by Year - {period}")
    axis.set_xlabel("Event year")
    axis.set_ylabel("Accident count")
plt.tight_layout()
plt.show()

## Target Distribution And Class Imbalance

Severity is the label used as `true_severity`. The before-2020 distribution drives model strategy. The after-2020 distribution is shown for drift awareness only.

In [ ]:
severity_counts = (
    eda_df.groupby(["period", "true_severity"])
    .size()
    .reset_index(name="count")
)
severity_counts["share"] = severity_counts["count"] / severity_counts.groupby("period")["count"].transform("sum")
severity_pivot = severity_counts.pivot(index="true_severity", columns="period", values=["count", "share"]).fillna(0)

display(severity_pivot)

imbalance_rows = []
for period, data in eda_df.groupby("period"):
    counts = data["true_severity"].value_counts().sort_index()
    imbalance_rows.append({
        "period": period,
        "majority_class": int(counts.idxmax()),
        "majority_count": int(counts.max()),
        "minority_class": int(counts.idxmin()),
        "minority_count": int(counts.min()),
        "majority_to_minority_ratio": round(float(counts.max() / counts.min()), 2),
    })
imbalance_table = pd.DataFrame(imbalance_rows)
display(imbalance_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    data = severity_counts[severity_counts["period"] == period].sort_values("true_severity")
    axis.pie(data["count"], labels=data["true_severity"], autopct="%1.1f%%", startangle=90)
    axis.set_title(f"Severity Share - {period}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    data = severity_counts[severity_counts["period"] == period]
    sns.barplot(data=data, x="true_severity", y="count", ax=axis, palette="viridis")
    axis.set_title(f"Severity Count - {period}")
    axis.set_xlabel("true_severity")
    axis.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Missing Values And Data Quality

Raw missing-value checks are useful for data engineering. Engineered missing-value checks are the model contract. The model should consume the engineered schema, not a raw concat with mismatched column names.

In [ ]:
engineered_columns = [
    "event_id", "event_year", "event_time", "true_severity", "lat", "lon",
    "hour", "day_of_week", "is_weekend", "is_rush_hour", "weather_code",
    "temperature_f", "humidity", "wind_speed_mph", "visibility_mi",
    "road_type_code", "is_junction", "has_traffic_signal", "is_crossing",
    "is_roundabout", "is_stop", "is_station", "is_railway", "is_night",
]

missing_tables = []
for period, data in [("before_2020_pretrain", before_df), ("from_2020_realtime", after_df)]:
    missing = data[engineered_columns].isna().sum().reset_index()
    missing.columns = ["feature", "missing_count"]
    missing["period"] = period
    missing["missing_share"] = missing["missing_count"] / len(data)
    missing_tables.append(missing)
missing_table = pd.concat(missing_tables, ignore_index=True)

display(missing_table.sort_values(["missing_share", "feature"], ascending=[False, True]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    plot_data = missing_table[missing_table["period"] == period].sort_values("missing_share", ascending=False).head(15)
    sns.barplot(data=plot_data, y="feature", x="missing_share", ax=axis, color="#F58518")
    axis.set_title(f"Top Engineered Missing Rates - {period}")
    axis.set_xlabel("Missing share")
    axis.set_ylabel("")
plt.tight_layout()
plt.show()

## Numeric Feature Distributions And Outliers

Weather variables are clipped in `processing.feature_engineering` to avoid physically impossible values. These histograms verify that training and realtime replay consume bounded values.

In [ ]:
numeric_features = ["temperature_f", "humidity", "wind_speed_mph", "visibility_mi", "lat", "lon"]
summary = (
    eda_df.groupby("period")[numeric_features]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(3)
)
display(summary)

for feature in numeric_features[:4]:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
    for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
        sns.histplot(eda_df.loc[eda_df["period"] == period, feature], bins=40, kde=True, ax=axis, color="#54A24B")
        axis.set_title(f"{feature} Histogram - {period}")
        axis.set_xlabel(feature)
        axis.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    plot_data = eda_df[eda_df["period"] == period][numeric_features[:4]].melt(var_name="feature", value_name="value")
    sns.boxplot(data=plot_data, x="feature", y="value", ax=axis)
    axis.set_title(f"Bounded Weather Feature Boxplots - {period}")
    axis.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Time-Based Accident Patterns

Rush-hour features should be justified by the data. Counts and average severity are shown by hour for both periods.

In [ ]:
hour_stats = (
    eda_df.groupby(["period", "hour"])
    .agg(accident_count=("event_id", "count"), avg_severity=("true_severity", "mean"), severe_rate=("true_severity", lambda s: (s >= 3).mean()))
    .reset_index()
)
display(hour_stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharex=True)
sns.lineplot(data=hour_stats, x="hour", y="accident_count", hue="period", marker="o", ax=axes[0])
axes[0].set_title("Accident Count by Hour")
axes[0].set_ylabel("Accident count")
sns.lineplot(data=hour_stats, x="hour", y="avg_severity", hue="period", marker="o", ax=axes[1])
axes[1].set_title("Average Severity by Hour")
axes[1].set_ylabel("Average true_severity")
plt.tight_layout()
plt.show()

rush_table = (
    eda_df.groupby(["period", "is_rush_hour", "is_weekend"])
    .agg(accident_count=("event_id", "count"), avg_severity=("true_severity", "mean"), severe_rate=("true_severity", lambda s: (s >= 3).mean()))
    .reset_index()
    .round(4)
)
display(rush_table)

## Weather And Road Context

Weather and road context may not have high linear correlation alone, but tree-based H2O models can combine them with location, time, and infrastructure flags.

In [ ]:
weather_stats = (
    eda_df.groupby(["period", "weather_code"])
    .agg(accident_count=("event_id", "count"), avg_severity=("true_severity", "mean"), severe_rate=("true_severity", lambda s: (s >= 3).mean()))
    .reset_index()
    .round(4)
)
road_stats = (
    eda_df.groupby(["period", "road_type_code"])
    .agg(accident_count=("event_id", "count"), avg_severity=("true_severity", "mean"), severe_rate=("true_severity", lambda s: (s >= 3).mean()))
    .reset_index()
    .round(4)
)

display(weather_stats)
display(road_stats)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
sns.barplot(data=weather_stats, x="weather_code", y="avg_severity", hue="period", ax=axes[0, 0])
axes[0, 0].set_title("Average Severity by Weather Code")
sns.barplot(data=weather_stats, x="weather_code", y="severe_rate", hue="period", ax=axes[0, 1])
axes[0, 1].set_title("Severe Rate by Weather Code")
sns.barplot(data=road_stats, x="road_type_code", y="avg_severity", hue="period", ax=axes[1, 0])
axes[1, 0].set_title("Average Severity by Road Type Code")
sns.barplot(data=road_stats, x="road_type_code", y="severe_rate", hue="period", ax=axes[1, 1])
axes[1, 1].set_title("Severe Rate by Road Type Code")
plt.tight_layout()
plt.show()

In [ ]:
road_flags = ["is_junction", "has_traffic_signal", "is_crossing", "is_roundabout", "is_stop", "is_station", "is_railway", "is_night"]
flag_rows = []
for period, data in eda_df.groupby("period"):
    for flag in road_flags:
        grouped = data.groupby(flag).agg(
            accident_count=("event_id", "count"),
            avg_severity=("true_severity", "mean"),
            severe_rate=("true_severity", lambda s: (s >= 3).mean()),
        ).reset_index()
        for _, row in grouped.iterrows():
            flag_rows.append({"period": period, "feature": flag, "value": int(row[flag]), "accident_count": int(row["accident_count"]), "avg_severity": row["avg_severity"], "severe_rate": row["severe_rate"]})
flag_table = pd.DataFrame(flag_rows).round(4)
display(flag_table)

fig, axes = plt.subplots(1, 2, figsize=(17, 5), sharey=True)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    plot_data = flag_table[(flag_table["period"] == period) & (flag_table["value"] == 1)].sort_values("severe_rate", ascending=False)
    sns.barplot(data=plot_data, y="feature", x="severe_rate", ax=axis, color="#B279A2")
    axis.set_title(f"Severe Rate When Flag Is Present - {period}")
    axis.set_xlabel("P(true_severity >= 3)")
    axis.set_ylabel("")
plt.tight_layout()
plt.show()

## Correlation Analysis

Correlation is calculated on engineered numeric features. The heatmap is visual, and the table ranks features by absolute correlation with `true_severity` for report writing.

In [ ]:
corr_features = [
    "true_severity", "lat", "lon", "hour", "day_of_week", "is_weekend", "is_rush_hour",
    "weather_code", "temperature_f", "humidity", "wind_speed_mph", "visibility_mi",
    "road_type_code", "is_junction", "has_traffic_signal", "is_crossing",
    "is_roundabout", "is_stop", "is_station", "is_railway", "is_night",
]

for period in ["before_2020_pretrain", "from_2020_realtime"]:
    corr = eda_df.loc[eda_df["period"] == period, corr_features].corr(numeric_only=True)
    severity_corr = (
        corr["true_severity"].drop("true_severity").sort_values(key=lambda s: s.abs(), ascending=False).reset_index()
    )
    severity_corr.columns = ["feature", "correlation_with_true_severity"]
    print(f"Top correlations for {period}")
    display(severity_corr.head(15))

    plt.figure(figsize=(13, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2, cbar_kws={"shrink": 0.8})
    plt.title(f"Correlation Heatmap - {period}")
    plt.tight_layout()
    plt.show()

## Spatial Sanity Check

The model consumes latitude and longitude. These plots confirm that sampled records remain in the expected US coordinate range after feature engineering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
for axis, period in zip(axes, ["before_2020_pretrain", "from_2020_realtime"]):
    plot_data = eda_df[eda_df["period"] == period].sample(min(50_000, (eda_df["period"] == period).sum()), random_state=RANDOM_SEED)
    scatter = axis.scatter(plot_data["lon"], plot_data["lat"], c=plot_data["true_severity"], s=2, alpha=0.35, cmap="viridis")
    axis.set_title(f"Sampled Accident Points - {period}")
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    axis.set_xlim(-130, -60)
    axis.set_ylim(20, 55)
fig.colorbar(scatter, ax=axes, label="true_severity", shrink=0.8)
plt.show()

## Feature And Modeling Decisions From Before-2020 Only

Use the after-2020 split for drift visibility and replay testing, not for initial model selection. Based on the before-2020 analysis:

- Keep the unified engineered schema from `build_features()`.
- Use H2O tree-based AutoML models instead of a linear-only model because correlations with severity are weak and interactions matter.
- Enable `balance_classes=True` and explicit bounded `class_sampling_factors` because severity 4 is rare.
- Report macro F1, weighted F1, per-class recall, and confusion matrix instead of accuracy alone.
- Keep clipped weather variables because they reduce outlier influence without removing valid accident rows.